In [7]:
import sys, nltk
print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)
assert sys.version_info >= (3, 10), "Se requiere Python 3.10 o superior"
for recurso in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(recurso, quiet=True)
print("Recursos de NLTK listos")

Python: 3.13.16
NLTK: 3.9.1
Recursos de NLTK listos


In [8]:
# -*- coding: utf-8 -*-
"""
Artroom 369 · Asistente de Orientación a Clientes (ARIA 369) — Versión 1.0
Chatbot basado en reglas con Python 3.10+ y NLTK.

Proyecto Fase 1 · MBIA2602 · Maestría en Inteligencia Artificial · Universidad Tecmilenio
Autora: Valeria Giselle Uribe Pérez

Arquitectura (4 componentes):
  1. Usuario            -> escribe una consulta en lenguaje natural.
  2. Chatbot            -> preprocesa con NLTK (tokenización, stopwords, stemming),
                           identifica la intención por reglas y elige la respuesta.
  3. Base de conocimiento -> BASE_CONOCIMIENTO + INTENCIONES (patrones y respuestas).
  4. Flujo conversacional -> estado de la sesión (última intención, acción pendiente,
                           datos del prospecto) que permite seguir la conversación.
"""

import re
import random
import unicodedata

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.metrics.distance import edit_distance


# ---------------------------------------------------------------------------
# 0. Recursos de NLTK
# ---------------------------------------------------------------------------
def preparar_nltk():
    """Descarga (una sola vez) los recursos de NLTK que usa el chatbot."""
    for recurso, ruta in [("punkt", "tokenizers/punkt"),
                          ("punkt_tab", "tokenizers/punkt_tab"),
                          ("stopwords", "corpora/stopwords")]:
        try:
            nltk.data.find(ruta)
        except LookupError:
            nltk.download(recurso, quiet=True)


# ---------------------------------------------------------------------------
# 1. BASE DE CONOCIMIENTO
#    Información institucional que el chatbot consulta para responder.
# ---------------------------------------------------------------------------
BASE_CONOCIMIENTO = {
    "empresa": "Artroom 369 Worldwide",
    "descripcion": ("consultoría boutique de comunicación con sede en la Ciudad de México, "
                    "fundada en 2010"),
    "servicios": {
        "Corporate & Industrial Communications":
            "comunicación interna y corporativa para empresas e industria: campañas internas, "
            "cultura organizacional, señalética de seguridad y mensajes para planta y oficinas.",
        "Branding":
            "creación y renovación de marcas: estrategia, naming, identidad visual y manual de marca.",
        "Brand Visual Strategy & Creative Direction":
            "dirección creativa y estrategia visual para que todas las piezas de la marca "
            "hablen el mismo idioma; es la especialidad diferenciadora de la firma.",
        "Content & Audiovisual Production":
            "contenido para redes y canales corporativos, video institucional, cápsulas y fotografía.",
        "Events":
            "concepto, diseño y producción de eventos corporativos: convenciones, lanzamientos, "
            "aniversarios y foros.",
        "Print & POP":
            "materiales impresos y de punto de venta: editoriales, displays, exhibidores y empaques.",
    },
    "no_ofrece": ["Relaciones Públicas", "BTL"],
    "marca_hermana": ("Atenea 369, especializada en diseño de interiores corporativos e "
                      "industriales"),
    "clientes": ["Uber", "Netflix", "Discovery", "Lenovo", "NuBank"],
    "proceso": [
        "1) Brief: entendemos tu objetivo, público, alcance y fechas.",
        "2) Propuesta: estrategia, concepto creativo, cronograma y cotización.",
        "3) Desarrollo: producción con rondas de revisión acordadas.",
        "4) Entrega y cierre: archivos finales, implementación y evaluación.",
    ],
    "datos_brief": ["nombre y empresa", "correo o teléfono", "servicio de interés",
                    "objetivo del proyecto", "fecha límite"],
}


# ---------------------------------------------------------------------------
# 2. INTENCIONES CONVERSACIONALES
#    Cada intención tiene: palabras clave (se comparan ya con stemming),
#    frases clave (se buscan en el texto normalizado) y respuestas asociadas.
# ---------------------------------------------------------------------------
def _lista_servicios():
    return "\n".join(f"   • {nombre}" for nombre in BASE_CONOCIMIENTO["servicios"])


INTENCIONES = {
    "saludo": {
        "palabras": ["hola", "buenas", "buenos", "saludos", "hey", "qué tal"],
        "frases": ["buenos dias", "buenas tardes", "buenas noches", "que tal"],
        "respuestas": [
            "Hola, soy ARIA, la asistente virtual de Artroom 369. Te oriento sobre nuestros "
            "servicios, el proceso de trabajo y cómo iniciar un proyecto. ¿Qué necesita tu marca?",
        ],
    },
    "despedida": {
        "palabras": ["adios", "bye", "chao", "hasta", "luego", "salir", "terminar"],
        "frases": ["hasta luego", "nos vemos", "es todo"],
        "respuestas": [
            "Gracias por escribir a Artroom 369. Cuando quieras retomar, aquí estaré.",
        ],
    },
    "agradecimiento": {
        "palabras": ["gracias", "agradezco", "amable", "excelente", "perfecto"],
        "frases": ["muchas gracias", "mil gracias"],
        "respuestas": [
            "Con gusto. ¿Hay algo más en lo que te pueda orientar?",
        ],
    },
    "servicios_general": {
        "palabras": ["servicios", "ofrecen", "hacen", "dedican", "catalogo", "especialidad",
                     "agencia", "consultoria"],
        "frases": ["que hacen", "a que se dedican", "que servicios", "que ofrecen"],
        "respuestas": [
            "Artroom 369 trabaja seis líneas de servicio:\n" + _lista_servicios() +
            "\n¿Sobre cuál quieres saber más?",
        ],
    },
    "comunicacion_corporativa": {
        "palabras": ["interna", "corporativa", "industrial", "colaboradores", "empleados",
                     "planta", "cultura", "organizacional", "señaletica"],
        "frases": ["comunicacion interna", "comunicacion corporativa", "comunicacion industrial"],
        "respuestas": [
            "Corporate & Industrial Communications: " +
            BASE_CONOCIMIENTO["servicios"]["Corporate & Industrial Communications"] +
            " ¿Tu necesidad es para oficinas, planta o ambas?",
        ],
    },
    "branding": {
        "palabras": ["branding", "marca", "logo", "logotipo", "identidad", "naming",
                     "rebranding", "manual"],
        "frases": ["crear mi marca", "nueva marca", "manual de marca", "identidad corporativa"],
        "respuestas": [
            "Branding: " + BASE_CONOCIMIENTO["servicios"]["Branding"] +
            " ¿Es una marca nueva o la renovación de una existente?",
        ],
    },
    "estrategia_visual": {
        "palabras": ["creativa", "creativo", "visual", "concepto", "direccion", "estetica",
                     "coherencia", "lineamientos"],
        "frases": ["direccion creativa", "estrategia visual", "brand visual"],
        "respuestas": [
            "Brand Visual Strategy & Creative Direction: " +
            BASE_CONOCIMIENTO["servicios"]["Brand Visual Strategy & Creative Direction"] +
            " ¿Quieres que te explique cómo arrancamos un proyecto así?",
        ],
    },
    "contenido_audiovisual": {
        "palabras": ["video", "videos", "contenido", "redes", "fotografia", "foto",
                     "audiovisual", "capsulas", "reel", "grabacion"],
        "frases": ["redes sociales", "video institucional", "produccion audiovisual"],
        "respuestas": [
            "Content & Audiovisual Production: " +
            BASE_CONOCIMIENTO["servicios"]["Content & Audiovisual Production"] +
            " ¿Para qué canal lo necesitas?",
        ],
    },
    "eventos": {
        "palabras": ["evento", "eventos", "convencion", "lanzamiento", "aniversario",
                     "foro", "congreso", "gala"],
        "frases": ["evento corporativo", "fiesta de fin de año"],
        "respuestas": [
            "Events: " + BASE_CONOCIMIENTO["servicios"]["Events"] +
            " ¿Para qué fecha y cuántos asistentes estimas?",
        ],
    },
    "impresos_pop": {
        "palabras": ["impresos", "impresion", "imprimir", "pop", "display", "displays",
                     "exhibidor", "empaque", "folleto", "catalogos", "lona"],
        "frases": ["punto de venta", "material pop", "print"],
        "respuestas": [
            "Print & POP: " + BASE_CONOCIMIENTO["servicios"]["Print & POP"] +
            " ¿Qué pieza tienes en mente y en qué cantidad?",
        ],
    },
    "proceso_brief": {
        "palabras": ["proceso", "pasos", "metodologia", "brief", "funciona", "trabajan",
                     "empezar", "iniciar", "arrancar"],
        "frases": ["como trabajan", "como empiezo", "que necesito", "como funciona"],
        "respuestas": [
            "Así trabajamos:\n   " + "\n   ".join(BASE_CONOCIMIENTO["proceso"]) +
            "\nPara el brief necesitamos: " + ", ".join(BASE_CONOCIMIENTO["datos_brief"]) + ".",
        ],
    },
    "cotizacion": {
        "palabras": ["precio", "precios", "costo", "costos", "cuesta", "cotizacion",
                     "cotizar", "presupuesto", "tarifa", "cobran"],
        "frases": ["cuanto cuesta", "cuanto cobran"],
        "respuestas": [
            "Cada cotización se construye a la medida del alcance, por eso no manejamos "
            "precios de lista. Con un brief breve preparamos una propuesta con cronograma y "
            "costo. ¿Quieres que agendemos una llamada para levantarlo?",
        ],
        "accion": "ofrecer_agenda",
    },
    "tiempos": {
        "palabras": ["tiempo", "tiempos", "tardan", "plazo", "entrega", "urgente",
                     "rapido", "cuando"],
        "frases": ["cuanto tardan", "cuanto tiempo", "fecha de entrega"],
        "respuestas": [
            "Los tiempos dependen del alcance y de las rondas de revisión. Tras el brief "
            "entregamos un cronograma con fechas comprometidas. Si tu proyecto es urgente, "
            "indícalo en el brief para priorizarlo.",
        ],
    },
    "experiencia_clientes": {
        "palabras": ["clientes", "portafolio", "experiencia", "trayectoria", "casos",
                     "trabajado", "referencias", "premios"],
        "frases": ["con quien han trabajado", "casos de exito"],
        "respuestas": [
            "Artroom 369 opera desde 2010 y ha colaborado con marcas como " +
            ", ".join(BASE_CONOCIMIENTO["clientes"]) +
            ". Si quieres ver casos de tu industria, podemos compartirlos en una llamada.",
        ],
        "accion": "ofrecer_agenda",
    },
    "contacto_agenda": {
        "palabras": ["contacto", "contactar", "llamada", "cita", "agendar", "reunion",
                     "asesor", "telefono", "correo", "whatsapp", "humano"],
        "frases": ["hablar con alguien", "quiero agendar", "una persona"],
        "respuestas": [
            "Con gusto te conecto con el equipo. Escríbeme tu nombre, empresa y correo "
            "electrónico, y un consultor de Artroom 369 te contactará para agendar.",
        ],
        "accion": "esperar_datos",
    },
    "servicio_no_ofrecido": {
        "palabras": ["rp", "btl", "activacion", "activaciones", "prensa", "medios",
                     "influencers", "volanteo", "sampling"],
        "frases": ["relaciones publicas", "boletin de prensa"],
        "respuestas": [
            "Ese servicio no forma parte de la oferta de Artroom 369: no trabajamos "
            "Relaciones Públicas ni BTL. Donde sí podemos sumar es en:\n" + _lista_servicios(),
        ],
    },
    "interiorismo_atenea": {
        "palabras": ["interiores", "interiorismo", "oficinas", "mobiliario", "layout",
                     "remodelacion", "espacios"],
        "frases": ["diseno de interiores", "diseno de oficinas"],
        "respuestas": [
            "El diseño de espacios lo atiende nuestra marca hermana, " +
            BASE_CONOCIMIENTO["marca_hermana"] +
            ". Si me dejas tus datos, canalizo tu solicitud con su equipo.",
        ],
        "accion": "esperar_datos",
    },
    "afirmacion": {
        "palabras": ["si", "claro", "ok", "va", "adelante", "dale", "sale"],
        "frases": ["por favor", "me interesa"],
        "respuestas": ["Perfecto. ¿Qué te gustaría saber?"],
    },
}

FALLBACK = [
    "No identifiqué tu consulta. Puedo ayudarte con: servicios, proceso de trabajo, "
    "cotizaciones, tiempos, experiencia o contacto con un consultor. ¿Sobre cuál te oriento?",
]

# Orden de desempate cuando dos intenciones obtienen el mismo puntaje
PRIORIDAD = ["servicio_no_ofrecido", "interiorismo_atenea", "contacto_agenda", "cotizacion",
             "tiempos", "proceso_brief", "comunicacion_corporativa", "branding",
             "estrategia_visual", "contenido_audiovisual", "eventos", "impresos_pop",
             "experiencia_clientes", "servicios_general", "despedida", "agradecimiento",
             "saludo", "afirmacion"]

# Intenciones que corresponden a una línea de servicio (para detectar consultas múltiples)
SERVICIOS_INTENCION = {
    "comunicacion_corporativa": "Corporate & Industrial Communications",
    "branding": "Branding",
    "estrategia_visual": "Brand Visual Strategy & Creative Direction",
    "contenido_audiovisual": "Content & Audiovisual Production",
    "eventos": "Events",
    "impresos_pop": "Print & POP",
}

PATRON_CORREO = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")


# ---------------------------------------------------------------------------
# 3. CHATBOT
# ---------------------------------------------------------------------------
class ChatbotArtroom:
    """Chatbot basado en reglas: preprocesamiento NLTK + puntaje por reglas + estado."""

    UMBRAL = 2  # puntaje mínimo para aceptar una intención

    def __init__(self, semilla=369):
        preparar_nltk()
        self.stemmer = SnowballStemmer("spanish")
        self.stop = {self._sin_acentos(p) for p in stopwords.words("spanish")}
        self.azar = random.Random(semilla)
        # Las palabras clave se guardan ya con stemming para compararlas con la entrada
        self.reglas = {}
        for intencion, datos in INTENCIONES.items():
            raices = {self.stemmer.stem(self._sin_acentos(p.lower())) for p in datos["palabras"]}
            frases = [self._sin_acentos(f.lower()) for f in datos["frases"]]
            self.reglas[intencion] = {"raices": raices, "frases": frases}
        self.reiniciar()

    # --- Flujo conversacional: estado de la sesión -------------------------
    def reiniciar(self):
        self.estado = {"ultima_intencion": None, "accion_pendiente": None,
                       "prospecto": {}, "turnos": 0}
        self._ultimos_puntajes = {}

    # --- Preprocesamiento con NLTK -----------------------------------------
    @staticmethod
    def _sin_acentos(texto):
        texto = unicodedata.normalize("NFD", texto)
        texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
        return texto.replace("ñ", "n")

    def normalizar(self, texto):
        """Minúsculas, sin acentos y sin las cláusulas negadas.

        Regla de negación: en cada cláusula (separada por coma, punto, etc.) se descarta
        todo lo que va después de "no". Así, "no quiero un logo, quiero un video" se
        interpreta como "quiero un video".
        """
        texto = self._sin_acentos(texto.lower())
        clausulas = re.split(r"[,.;:!?¿¡]", texto)
        conservadas = []
        for c in clausulas:
            palabras = c.split()
            if "no" in palabras:
                palabras = palabras[:palabras.index("no")]
            conservadas.append(" ".join(palabras))
        return " , ".join(c for c in conservadas if c)

    def preprocesar(self, texto):
        """Tokeniza, elimina stopwords y signos, y aplica stemming (NLTK)."""
        tokens = word_tokenize(self.normalizar(texto), language="spanish")
        tokens = [t for t in tokens if t.isalpha()]
        # Las palabras cortas de respuesta ("si", "ok") se conservan aunque sean stopwords
        utiles = [t for t in tokens if t not in self.stop or t in {"si", "no", "ok"}]
        return [self.stemmer.stem(t) for t in utiles]

    # --- Identificación de intención por reglas ----------------------------
    def puntuar(self, texto):
        normal = self.normalizar(texto)
        raices = self.preprocesar(texto)
        puntajes = {}
        for intencion, regla in self.reglas.items():
            puntos = 0
            for frase in regla["frases"]:                 # regla 1: frase completa (+3)
                if frase in normal:
                    puntos += 3
            for r in raices:
                if r in regla["raices"]:                   # regla 2: raíz exacta (+2)
                    puntos += 2
                elif len(r) >= 5 and any(                  # regla 3: error de dedo (+2)
                        len(k) >= 5 and edit_distance(r, k) == 1 for k in regla["raices"]):
                    puntos += 2
            if puntos:
                puntajes[intencion] = puntos
        return puntajes

    def identificar_intencion(self, texto):
        puntajes = self.puntuar(texto)
        # "afirmacion" solo cuenta si no hay otra intención con contenido en el mensaje
        if len(puntajes) > 1:
            puntajes.pop("afirmacion", None)
        self._ultimos_puntajes = puntajes
        if not puntajes:
            return "fallback", 0
        mejor = max(puntajes.values())
        if mejor < self.UMBRAL:
            return "fallback", mejor
        empatadas = [i for i, p in puntajes.items() if p == mejor]
        empatadas.sort(key=lambda i: PRIORIDAD.index(i))
        return empatadas[0], mejor

    # --- Generación de respuesta -------------------------------------------
    def responder(self, texto):
        self.estado["turnos"] += 1

        # Regla de captura de datos: si el usuario escribe un correo, se registra el prospecto
        correo = PATRON_CORREO.search(texto)
        if correo:
            self.estado["prospecto"]["correo"] = correo.group()
            self.estado["accion_pendiente"] = None
            self.estado["ultima_intencion"] = "registro_datos"
            return ("registro_datos", 5,
                    f"Registré tu correo ({correo.group()}). Un consultor de Artroom 369 te "
                    "escribirá para agendar la llamada y levantar el brief. ¿Algo más?")

        intencion, puntaje = self.identificar_intencion(texto)

        # Regla de contexto: un "sí" después de ofrecer agenda lleva al contacto
        if intencion == "afirmacion" and self.estado["accion_pendiente"] == "ofrecer_agenda":
            intencion = "contacto_agenda"

        if intencion == "fallback":
            respuesta = self.azar.choice(FALLBACK)
        else:
            respuesta = self.azar.choice(INTENCIONES[intencion]["respuestas"])

        # Regla de consulta múltiple: si el mensaje menciona varias líneas de servicio,
        # se responde la principal y se registran las demás para integrarlas en el brief
        adicionales = [SERVICIOS_INTENCION[i] for i, p in self._ultimos_puntajes.items()
                       if i in SERVICIOS_INTENCION and i != intencion and p >= self.UMBRAL]
        if intencion != "fallback" and adicionales:
            self.estado["prospecto"]["servicios"] = adicionales
            respuesta += ("\n   También detecté interés en: " + ", ".join(adicionales) +
                          ". Podemos integrarlo todo en un mismo brief.")

        self.estado["ultima_intencion"] = intencion
        self.estado["accion_pendiente"] = INTENCIONES.get(intencion, {}).get("accion")
        return intencion, puntaje, respuesta


# ---------------------------------------------------------------------------
# 4. Utilidades para pruebas y uso interactivo
# ---------------------------------------------------------------------------
def conversar(bot, mensajes, titulo=""):
    """Ejecuta una conversación completa y devuelve el registro (evidencia)."""
    bot.reiniciar()
    registro = []
    if titulo:
        print(f"\n{'=' * 72}\n{titulo}\n{'=' * 72}")
    for m in mensajes:
        intencion, puntaje, respuesta = bot.responder(m)
        registro.append((m, intencion, puntaje, respuesta))
        print(f"Usuario : {m}")
        print(f"[intención: {intencion} | puntaje: {puntaje}]")
        print(f"ARIA    : {respuesta}\n")
    return registro


def chat_interactivo():
    bot = ChatbotArtroom()
    print("ARIA · Artroom 369 (escribe 'salir' para terminar)")
    while True:
        texto = input("Tú: ").strip()
        if not texto:
            continue
        intencion, _, respuesta = bot.responder(texto)
        print("ARIA:", respuesta)
        if intencion == "despedida":
            break

In [9]:
bot = ChatbotArtroom()
print("Intenciones definidas:", len(INTENCIONES))
print("Ejemplo de preprocesamiento:")
ejemplo = "¿Cuánto cuestan los videos institucionales?"
print("  Entrada :", ejemplo)
print("  Tokens  :", word_tokenize(bot.normalizar(ejemplo), language="spanish"))
print("  Raíces  :", bot.preprocesar(ejemplo))
print("  Puntajes:", bot.puntuar(ejemplo))

Intenciones definidas: 18
Ejemplo de preprocesamiento:
  Entrada : ¿Cuánto cuestan los videos institucionales?
  Tokens  : ['cuanto', 'cuestan', 'los', 'videos', 'institucionales']
  Raíces  : ['cuant', 'cuest', 'vide', 'institucional']
  Puntajes: {'contenido_audiovisual': 2, 'cotizacion': 5, 'tiempos': 2}


In [10]:
CONVERSACIONES = [
    ("Conversación 1 · Prospecto explora servicios",
     ["Hola, buenas tardes", "¿Qué servicios ofrecen?", "Muchas gracias, hasta luego"]),
    ("Conversación 2 · Director de RR. HH. busca comunicación interna",
     ["Buenos días", "Necesito una campaña de comunicación interna para los empleados de planta",
      "¿Cómo trabajan?", "Gracias"]),
    ("Conversación 3 · Startup quiere crear su marca y cotizar",
     ["Queremos crear nuestra marca y el logotipo", "¿Cuánto cuesta?", "Sí, por favor",
      "laura.mendez@empresa.com"]),
    ("Conversación 4 · Gerente de marketing busca dirección creativa",
     ["Busco dirección creativa para que todas nuestras piezas sean coherentes",
      "¿Con quién han trabajado?", "Excelente, es todo"]),
    ("Conversación 5 · Producción de video y redes sociales",
     ["Hola", "Necesitamos un video institucional y contenido para redes sociales",
      "¿Cuánto tiempo tardan?"]),
    ("Conversación 6 · Evento corporativo",
     ["Estamos organizando un evento corporativo por el aniversario de la empresa",
      "¿Qué necesito para empezar?", "Quiero agendar una reunión con un asesor"]),
    ("Conversación 7 · Material de punto de venta",
     ["Hola, ¿hacen displays y exhibidores para punto de venta?",
      "¿Cuál es el precio de un exhibidor?", "No por ahora, gracias"]),
    ("Conversación 8 · Solicitud fuera de oferta (RP y BTL)",
     ["¿Hacen relaciones públicas?", "¿Y activaciones BTL?",
      "Entonces me interesa branding"]),
    ("Conversación 9 · Solicitud de interiorismo (marca hermana)",
     ["Necesito diseño de interiores para nuestras oficinas",
      "Mi correo es compras@industrias-norte.mx", "Gracias, adiós"]),
    ("Conversación 10 · Errores de escritura (tolerancia con edit_distance)",
     ["Quiero una cotisacion", "¿Ofresen fotografia?", "Bye"]),
    ("Conversación 11 · Consulta fuera de dominio (fallback)",
     ["¿Qué clima hará mañana?", "¿Venden computadoras?", "Ok, ¿qué servicios tienen?"]),
    ("Conversación 12 · Negación y consulta múltiple",
     ["No quiero un logo, quiero un video",
      "Necesito branding, un evento y material impreso, ¿cuánto cuesta todo?",
      "Sí, agendemos", "Gracias, es todo"]),
]

# Pruebas de límites: entradas diseñadas para encontrar dónde falla la versión 1.0
PRUEBAS_LIMITE = [
    "Quiero que la gente reconozca mi empresa",          # paráfrasis sin palabras clave
    "¿Cuánto cuesta el branding?",                        # dos intenciones en una pregunta
    "¿Y eso cuánto tarda?",                               # referencia a un turno anterior
    "No me gustó mi logo actual, ¿pueden renovarlo?",    # negación que no excluye el tema
    "Do you offer video production?",                     # consulta en inglés
]

In [11]:
from datetime import datetime
print("Ejecución:", datetime.now().strftime("%Y-%m-%d %H:%M"))
resultados = {}
for titulo, mensajes in CONVERSACIONES:
    resultados[titulo] = conversar(bot, mensajes, titulo)

Ejecución: 2026-10-26 16:59

Conversación 1 · Prospecto explora servicios
Usuario : Hola, buenas tardes
[intención: saludo | puntaje: 7]
ARIA    : Hola, soy ARIA, la asistente virtual de Artroom 369. Te oriento sobre nuestros servicios, el proceso de trabajo y cómo iniciar un proyecto. ¿Qué necesita tu marca?

Usuario : ¿Qué servicios ofrecen?
[intención: servicios_general | puntaje: 7]
ARIA    : Artroom 369 trabaja seis líneas de servicio:
   • Corporate & Industrial Communications
   • Branding
   • Brand Visual Strategy & Creative Direction
   • Content & Audiovisual Production
   • Events
   • Print & POP
¿Sobre cuál quieres saber más?

Usuario : Muchas gracias, hasta luego
[intención: despedida | puntaje: 5]
ARIA    : Gracias por escribir a Artroom 369. Cuando quieras retomar, aquí estaré.


Conversación 2 · Director de RR. HH. busca comunicación interna
Usuario : Buenos días
[intención: saludo | puntaje: 5]
ARIA    : Hola, soy ARIA, la asistente virtual de Artroom 369. Te oriento

In [12]:
import collections
turnos = [t for r in resultados.values() for t in r]
conteo = collections.Counter(t[1] for t in turnos)
print(f"Conversaciones: {len(resultados)} | Turnos: {len(turnos)}")
print(f"Intenciones distintas reconocidas: {len([i for i in conteo if i != 'fallback'])}")
for intencion, n in conteo.most_common():
    print(f"  {intencion:<26}{n}")

Conversaciones: 12 | Turnos: 39
Intenciones distintas reconocidas: 18
  despedida                 5
  cotizacion                4
  saludo                    3
  contacto_agenda           3
  contenido_audiovisual     3
  servicios_general         2
  proceso_brief             2
  agradecimiento            2
  branding                  2
  registro_datos            2
  servicio_no_ofrecido      2
  fallback                  2
  comunicacion_corporativa  1
  estrategia_visual         1
  experiencia_clientes      1
  tiempos                   1
  eventos                   1
  impresos_pop              1
  interiorismo_atenea       1


In [13]:
for m in PRUEBAS_LIMITE:
    conversar(bot, [m])

Usuario : Quiero que la gente reconozca mi empresa
[intención: impresos_pop | puntaje: 2]
ARIA    : Print & POP: materiales impresos y de punto de venta: editoriales, displays, exhibidores y empaques. ¿Qué pieza tienes en mente y en qué cantidad?

Usuario : ¿Cuánto cuesta el branding?
[intención: cotizacion | puntaje: 5]
ARIA    : Cada cotización se construye a la medida del alcance, por eso no manejamos precios de lista. Con un brief breve preparamos una propuesta con cronograma y costo. ¿Quieres que agendemos una llamada para levantarlo?
   También detecté interés en: Branding. Podemos integrarlo todo en un mismo brief.

Usuario : ¿Y eso cuánto tarda?
[intención: tiempos | puntaje: 4]
ARIA    : Los tiempos dependen del alcance y de las rondas de revisión. Tras el brief entregamos un cronograma con fechas comprometidas. Si tu proyecto es urgente, indícalo en el brief para priorizarlo.

Usuario : No me gustó mi logo actual, ¿pueden renovarlo?
[intención: fallback | puntaje: 0]
ARIA    